In [47]:
%pip install Sastrawi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
#1. Import library
import pandas as pd
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [49]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

print(stemmer.stem("mengerjakan"))

kerja


In [55]:
import pandas as pd
#2. Load Dataset
data = pd.read_csv("dataset.csv")
text_column = "content_article"

# Cek apakah dataset berhasil dimuat
print("Ukuran dataset:", data.shape)
print("Daftar Kolom:", data.columns.tolist())
print("Jumlah data:", len(data))

data[[text_column]].head(10)

print(data.head())
print("Kolom teks yang digunakan:", text_column)


Ukuran dataset: (3376, 8)
Daftar Kolom: ['date_publication', 'media_source', 'title_article', 'content_article', 'url_article', 'sentiment_label', 'polarity_score', 'ai_confidence']
Jumlah data: 3376
  date_publication media_source  \
0       2026-01-01       Kontan   
1       2026-01-01       Kontan   
2       2026-01-01       Kontan   
3       2026-01-01       Kontan   
4       2026-01-01       Kontan   

                                       title_article  \
0  Harga Emas Antam Hari Ini (28/3) Naik Rp 27.00...   
1  Kejagung Tetapkan Samin Tan Tersangka, Ini Kas...   
2  Asing Net Sell Rp 20,38 Triliun, Cek Saham yan...   
3                          Berburu Emas di Pagi Buta   
4  Hampir Rampung, ESDM Restui RKAB Batubara 580 ...   

                                     content_article  \
0  Reporter: Noverius Laoli | Editor: Noverius La...   
1  Reporter: kompas.com | Editor: Noverius Laoli ...   
2  Reporter: Noverius Laoli | Editor: Noverius La...   
3                      Forma

In [ ]:
#3.Case Folding
def case_folding(text):
    return str(text).lower()

# Contoh
text = "Saya Belajar NLP"
print(case_folding(text))
# Output: saya belajar nlp

saya belajar nlp


In [ ]:
#4. Menghapus URL
def remove_url(text):
    return re.sub(r"http\S+|www\S+", "", text)

# Contoh
text = "Kunjungi https://example.com sekarang"
print(remove_url(text))
# Output: Kunjungi  sekarang

Kunjungi  sekarang


In [ ]:
#5. Menghapus Mention
def remove_mention(text):
    return re.sub(r"@\w+", "", text)

# Contoh
text = "@user1 produk ini bagus"
print(remove_mention(text))

 produk ini bagus


In [56]:
#6. Menghapus hashtag
def remove_hashtag(text):
    return re.sub(r"#\w+", "", text)

#Contoh
text = "Pulau Lombok #indonesia"
print(remove_hashtag(text))

Pulau Lombok 


In [57]:
#7. Menghapus Angka
def remove_number(text):
    return re.sub(r"\d+", "", text)

#Contoh
text = "Saya memiliki 2 kucing dan 3 anjing"
print(remove_number(text))

Saya memiliki  kucing dan  anjing


In [58]:
#8. Menghapus Punctuation
def remove_punctuation(text):
    return re.sub(r"[^\w\s]", " ", text)

#Contoh
text = "Halo, apa kabar?"
print(remove_punctuation(text))

Halo  apa kabar 


In [59]:
#9. Menghapus Whitespace
def remove_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

#Contoh
text = "ArithmeticError   adalah   kesalahan  yang  terjadi  saat  melakukan  operasi  aritmatika."
print(remove_whitespace(text))

ArithmeticError adalah kesalahan yang terjadi saat melakukan operasi aritmatika.


In [62]:
#10. Fungsi Cleaning Lengkap
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = remove_url(text)
    text = remove_mention(text)
    text = remove_hashtag(text)
    text = remove_number(text)
    text = remove_punctuation(text)
    text = clean_whitespace(text)
    return text

#menerapkan ke dataset
data["clean_text"] = data[text_column].apply(clean_text)
data[[text_column, "clean_text"]].head()

,content_article,clean_text
0,Reporter: Noverius Laoli | Editor: Noverius La...,reporter noverius laoli editor noverius laoli ...
1,Reporter: kompas.com | Editor: Noverius Laoli ...,reporter kompas com editor noverius laoli kont...
2,Reporter: Noverius Laoli | Editor: Noverius La...,reporter noverius laoli editor noverius laoli ...
3,Format Non-Text/Gagal Ekstrak,format non text gagal ekstrak
4,Reporter: Sabrina Rhamadanty | Editor: Ignatia...,reporter sabrina rhamadanty editor ignatia mar...


In [63]:
#11. Stopword Removal
stop_words = set(stopwords.words("indonesian"))

def remove_stopwords(tokens):
    return [
        token for token in tokens
        if token not in stop_words
    ]

# Contoh
tokens = ["saya", "sedang", "belajar", "nlp", "yang", "menarik"]
print(remove_stopwords(tokens))

['belajar', 'nlp', 'menarik']


In [64]:
#12. Stemming dengan Sastrawi
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

# Contoh
text = "Mahasiswa sedang mempelajari pemrograman."
print(stemmer.stem(text))

mahasiswa sedang ajar pemrograman


In [ ]:
#13. Pipeline Preprocessing Lengkap
def preprocessing_pipeline(text):
    # 1. Cleaning
    text = clean_text(text)

    # 2. Tokenization
    tokens = word_tokenize(text)

    # 3. Stopword removal
    tokens = remove_stopwords(tokens)

    # 4. Stemming
    tokens = stem_tokens(tokens)

    return tokens

#Menerapkan preprocessing secara bertahap
import time

text_column = "content_article"
batch_size = 1000
processed_tokens = []

start_time = time.time()

for start in range(0, len(data), batch_size):
    end = start + batch_size
    
    batch = data[text_column].iloc[start:end]
    hasil = batch.apply(preprocessing_pipeline)
    
    processed_tokens.extend(hasil.tolist())
    
    print(f"Processed {end} of {len(data)} selesai diproses.")
    
#Menyimpan hasil tokenisasi ke preprocessing
data["processed_tokens"] = processed_tokens

print("\nPreprocessing selesai!")
print(f"Waktu yang dibutuhkan: {time.time() - start_time:.2f} detik")

#Menampilkan 10 hasil pertama
print(data[["content_article", "processed_tokens"]].head(10))

# Menerapkan ke dataset
data["processed_tokens"] = data[text_column].apply(preprocessing_pipeline)
print(data[["content_article", "processed_tokens"]].head(10))

Processed 1000 of 3376 selesai diproses.
Processed 2000 of 3376 selesai diproses.
Processed 3000 of 3376 selesai diproses.
Processed 4000 of 3376 selesai diproses.

Preprocessing selesai!
Waktu yang dibutuhkan: 13.35 detik
                                     content_article  \
0  Reporter: Noverius Laoli | Editor: Noverius La...   
1  Reporter: kompas.com | Editor: Noverius Laoli ...   
2  Reporter: Noverius Laoli | Editor: Noverius La...   
3                      Format Non-Text/Gagal Ekstrak   
4  Reporter: Sabrina Rhamadanty | Editor: Ignatia...   
5  Reporter: kompas.com | Editor: Noverius Laoli ...   
6  KONTAN.CO.ID - Yuk lihat harga emas hari ini d...   
7  Reporter: Dimas Andi | Editor: Tri Sulistiowat...   
8  KONTAN.CO.ID - Yuk lihat harga emas Antam loga...   
9  Sumber: Reuters | Editor: Khomarul Hidayat KON...   

                                    processed_tokens  
0  [reporter, noverius, laoli, editor, noverius, ...  
1  [reporter, kompas, com, editor, noverius, laol.

In [74]:
#14. Menggabungkan Token Kembali
data["processed_text"] = data["processed_tokens"].apply(
    lambda tokens: " ".join(tokens)
)

#Menampilkan hasil preprocessing
print(data[["content_article", "processed_tokens", "processed_text"]].head(10))

                                     content_article  \
0  Reporter: Noverius Laoli | Editor: Noverius La...   
1  Reporter: kompas.com | Editor: Noverius Laoli ...   
2  Reporter: Noverius Laoli | Editor: Noverius La...   
3                      Format Non-Text/Gagal Ekstrak   
4  Reporter: Sabrina Rhamadanty | Editor: Ignatia...   
5  Reporter: kompas.com | Editor: Noverius Laoli ...   
6  KONTAN.CO.ID - Yuk lihat harga emas hari ini d...   
7  Reporter: Dimas Andi | Editor: Tri Sulistiowat...   
8  KONTAN.CO.ID - Yuk lihat harga emas Antam loga...   
9  Sumber: Reuters | Editor: Khomarul Hidayat KON...   

                                    processed_tokens  \
0  [reporter, noverius, laoli, editor, noverius, ...   
1  [reporter, kompas, com, editor, noverius, laol...   
2  [reporter, noverius, laoli, editor, noverius, ...   
3                [format, non, text, gagal, ekstrak]   
4  [reporter, sabrina, rhamadanty, editor, ignati...   
5  [reporter, kompas, com, editor, noverius, la

In [75]:
#15. Perbandingan Raw vs Processed
for i in range(5):
    print("RAW:")
    print(data.iloc[i][text_column])

    print("\nPROCESSED:")
    print(data.iloc[i]["processed_text"])

    print("=" * 80)

RAW:
Reporter: Noverius Laoli | Editor: Noverius Laoli KONTAN.CO.ID - JAKARTA.  Harga emas batangan bersertifikat Antam keluaran Logam Mulia PT Aneka Tambang Tbk (ANTM) naik pada Sabtu (28/3/2026). Mengutip situs Logam Mulia, harga pecahan satu gram emas Antam berada di Rp 2.837.000. Harga emas Antam itu naik dibandingkan dengan harga pada Jumat (27/3/2026) yang berada di level Rp 2.810.000 per gram. Sementara harga buyback emas Antam berada di level Rp 2.461.000 per gram. Harga tersebut naik Rp 47.000 jika dibandingkan dengan harga buyback pada Jumat (27/3/2026) yang juga ada di Rp 2.414.000 per gram. Berikut harga emas batangan Antam dalam pecahan lainnya per Sabtu (28/3/2026) dan belum termasuk pajak:  Harga emas 0,5 gram: Rp 1.468.000 Harga emas 1 gram: Rp 2.837.000 Harga emas 5 gram: Rp 13.960.000 Harga emas 10 gram: Rp 27.865.000 Harga emas 25 gram: Rp 69.537.000 Harga emas 50 gram: Rp 138.995.000 Harga emas 100 gram: Rp 277.912.000 Harga emas 250 gram: Rp 694.515.000 Harga emas 

In [76]:
#16. Menyimpan Data Hasil Preprocessing
output_columns = [text_column, "clean_text", "processed_text"]

data[output_columns].to_csv("dataset_processed.csv", index=False)

print("Data berhasil disimpan.")

Data berhasil disimpan.


In [77]:
# Membuat kolom hasil cleaning
data["clean_text"] = data[text_column].apply(clean_text)

# Cek hasil
data[[text_column, "clean_text"]].head(10)

,content_article,clean_text
0,Reporter: Noverius Laoli | Editor: Noverius La...,reporter noverius laoli editor noverius laoli ...
1,Reporter: kompas.com | Editor: Noverius Laoli ...,reporter kompas com editor noverius laoli kont...
2,Reporter: Noverius Laoli | Editor: Noverius La...,reporter noverius laoli editor noverius laoli ...
3,Format Non-Text/Gagal Ekstrak,format non text gagal ekstrak
4,Reporter: Sabrina Rhamadanty | Editor: Ignatia...,reporter sabrina rhamadanty editor ignatia mar...
5,Reporter: kompas.com | Editor: Noverius Laoli ...,reporter kompas com editor noverius laoli kont...
6,KONTAN.CO.ID - Yuk lihat harga emas hari ini d...,kontan co id yuk lihat harga emas hari ini di ...
7,Reporter: Dimas Andi | Editor: Tri Sulistiowat...,reporter dimas andi editor tri sulistiowati ko...
8,KONTAN.CO.ID - Yuk lihat harga emas Antam loga...,kontan co id yuk lihat harga emas antam logam ...
9,Sumber: Reuters | Editor: Khomarul Hidayat KON...,sumber reuters editor khomarul hidayat kontan ...


In [ ]:
#E. TUGAS EKSPERIMEN: TABEL PERBANDINGAN 20 DATA
# Muat ulang data asli (supaya Raw Text benar-benar mentah)
raw = pd.read_csv("dataset.csv")

# Buang placeholder "Gagal Ekstrak", ambil 20 dokumen acak
sampel = raw[raw[text_column] != "Format Non-Text/Gagal Ekstrak"].sample(20, random_state=42)

def potong(teks, n=25):                  # 25 kata pertama agar tabel terbaca
    return " ".join(str(teks).split()[:n])

baris = []
for teks in sampel[text_column]:
    s = potong(teks)
    lower  = case_folding(s)
    clean  = clean_text(s)
    token  = word_tokenize(clean)
    stop   = remove_stopwords(token)
    stem   = stem_tokens(stop)
    baris.append({
        "Raw Text": s,
        "Setelah Lowercase": lower,
        "Setelah Cleaning": clean,
        "Setelah Tokenisasi": token,
        "Setelah Stopword": stop,
        "Setelah Stemming": stem,
    })

tabel_perbandingan = pd.DataFrame(baris)
tabel_perbandingan.to_csv("tabel_perbandingan.csv", index=False)
tabel_perbandingan.head(20)

,Raw Text,Setelah Lowercase,Setelah Cleaning,Setelah Tokenisasi,Setelah Stopword,Setelah Stemming
0,"Jakarta, CNBC Indonesia — Otoritas Jasa Keuang...","jakarta, cnbc indonesia — otoritas jasa keuang...",jakarta cnbc indonesia otoritas jasa keuangan ...,"[jakarta, cnbc, indonesia, otoritas, jasa, keu...","[jakarta, cnbc, indonesia, otoritas, jasa, keu...","[jakarta, cnbc, indonesia, otoritas, jasa, uan..."
1,"Bisnis.com, JAKARTA – Transaksi jual investor ...","bisnis.com, jakarta – transaksi jual investor ...",bisnis com jakarta transaksi jual investor asi...,"[bisnis, com, jakarta, transaksi, jual, invest...","[bisnis, com, jakarta, transaksi, jual, invest...","[bisnis, com, jakarta, transaksi, jual, invest..."
2,Reporter: Yudho Winarto | Editor: Yudho Winart...,reporter: yudho winarto | editor: yudho winart...,reporter yudho winarto editor yudho winarto ko...,"[reporter, yudho, winarto, editor, yudho, wina...","[reporter, yudho, winarto, editor, yudho, wina...","[reporter, yudho, winarto, editor, yudho, wina..."
3,"Jakarta, CNBC Indonesia- Indeks harga saham ga...","jakarta, cnbc indonesia- indeks harga saham ga...",jakarta cnbc indonesia indeks harga saham gabu...,"[jakarta, cnbc, indonesia, indeks, harga, saha...","[jakarta, cnbc, indonesia, indeks, harga, saha...","[jakarta, cnbc, indonesia, indeks, harga, saha..."
4,Reporter: Alya Fathinah | Editor: Avanty Nurdi...,reporter: alya fathinah | editor: avanty nurdi...,reporter alya fathinah editor avanty nurdiana ...,"[reporter, alya, fathinah, editor, avanty, nur...","[reporter, alya, fathinah, editor, avanty, nur...","[reporter, alya, fathinah, editor, avanty, nur..."
5,Reporter: Avanty Nurdiana | Editor: Avanty Nur...,reporter: avanty nurdiana | editor: avanty nur...,reporter avanty nurdiana editor avanty nurdian...,"[reporter, avanty, nurdiana, editor, avanty, n...","[reporter, avanty, nurdiana, editor, avanty, n...","[reporter, avanty, nurdiana, editor, avanty, n..."
6,"Bisnis.com, JAKARTA —Indeks harga saham gabung...","bisnis.com, jakarta —indeks harga saham gabung...",bisnis com jakarta indeks harga saham gabungan...,"[bisnis, com, jakarta, indeks, harga, saham, g...","[bisnis, com, jakarta, indeks, harga, saham, g...","[bisnis, com, jakarta, indeks, harga, saham, g..."
7,"Bisnis.com, JAKARTA — Indeks Harga Saham Gabun...","bisnis.com, jakarta — indeks harga saham gabun...",bisnis com jakarta indeks harga saham gabungan...,"[bisnis, com, jakarta, indeks, harga, saham, g...","[bisnis, com, jakarta, indeks, harga, saham, g...","[bisnis, com, jakarta, indeks, harga, saham, g..."
8,"Bisnis.com, JAKARTA – Indeks harga saham gabun...","bisnis.com, jakarta – indeks harga saham gabun...",bisnis com jakarta indeks harga saham gabungan...,"[bisnis, com, jakarta, indeks, harga, saham, g...","[bisnis, com, jakarta, indeks, harga, saham, g...","[bisnis, com, jakarta, indeks, harga, saham, g..."
9,"Jakarta, CNBC Indonesia — Indeks Harga Saham G...","jakarta, cnbc indonesia — indeks harga saham g...",jakarta cnbc indonesia indeks harga saham gabu...,"[jakarta, cnbc, indonesia, indeks, harga, saha...","[jakarta, cnbc, indonesia, indeks, harga, saha...","[jakarta, cnbc, indonesia, indeks, harga, saha..."


In [79]:
import os
import pandas as pd

file_final = "dataset_processed.csv"

print("File ditemukan:", os.path.exists(file_final))

if os.path.exists(file_final):
    cek = pd.read_csv(file_final, low_memory=False)
    print("Jumlah baris:", len(cek))
    print("Kolom tersedia:", cek.columns.tolist())
    display(cek[["content_article", "processed_text"]].head(5))

File ditemukan: True
Jumlah baris: 3376
Kolom tersedia: ['content_article', 'clean_text', 'processed_text']


,content_article,processed_text
0,Reporter: Noverius Laoli | Editor: Noverius La...,reporter noverius laoli editor noverius laoli ...
1,Reporter: kompas.com | Editor: Noverius Laoli ...,reporter kompas com editor noverius laoli kont...
2,Reporter: Noverius Laoli | Editor: Noverius La...,reporter noverius laoli editor noverius laoli ...
3,Format Non-Text/Gagal Ekstrak,format non text gagal ekstrak
4,Reporter: Sabrina Rhamadanty | Editor: Ignatia...,reporter sabrina rhamadanty editor ignatia mar...
